In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report


In [2]:
# Load only 100,000 rows to speed up your initial EDA
df = pd.read_csv('../data/PS_20174392719_1491204439457_log.csv', nrows=100000)

In [3]:
print(df.head())

   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  


In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   step            100000 non-null  int64  
 1   type            100000 non-null  object 
 2   amount          100000 non-null  float64
 3   nameOrig        100000 non-null  object 
 4   oldbalanceOrg   100000 non-null  float64
 5   newbalanceOrig  100000 non-null  float64
 6   nameDest        100000 non-null  object 
 7   oldbalanceDest  100000 non-null  float64
 8   newbalanceDest  100000 non-null  float64
 9   isFraud         100000 non-null  int64  
 10  isFlaggedFraud  100000 non-null  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 8.4+ MB
None


In [5]:
# Create the 'error_orig' feature
# If this is not zero, it means the balance didn't change correctly - a huge red flag!
df['error_orig'] = df['oldbalanceOrg'] - df['amount'] - df['newbalanceOrig']

# Create the 'error_dest' feature
# Similar logic for the destination account
df['error_dest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']

# Check the first few rows to see your new columns
print(df[['amount', 'oldbalanceOrg', 'newbalanceOrig', 'error_orig']].head())

     amount  oldbalanceOrg  newbalanceOrig  error_orig
0   9839.64       170136.0       160296.36         0.0
1   1864.28        21249.0        19384.72         0.0
2    181.00          181.0            0.00         0.0
3    181.00          181.0            0.00         0.0
4  11668.14        41554.0        29885.86         0.0


In [6]:
# Compare the average 'error_orig' for Fraud vs. Non-Fraud
print(df.groupby('isFraud')['error_orig'].mean())

isFraud
0   -190106.328022
1    -17948.772759
Name: error_orig, dtype: float64


In [7]:
# 1. Drop columns that are text-based or don't help the model (like 'nameOrig' and 'nameDest')
# In a real bank, names don't tell the model if it's fraud—the patterns do.
df = df.drop(['nameOrig', 'nameDest'], axis=1)

# 2. Convert the 'type' column (which is text) into numerical format (0, 1, 2, etc.)
df = pd.get_dummies(df, columns=['type'])

# 3. Check the new structure
print(df.head())


   step    amount  oldbalanceOrg  newbalanceOrig  oldbalanceDest  \
0     1   9839.64       170136.0       160296.36             0.0   
1     1   1864.28        21249.0        19384.72             0.0   
2     1    181.00          181.0            0.00             0.0   
3     1    181.00          181.0            0.00         21182.0   
4     1  11668.14        41554.0        29885.86             0.0   

   newbalanceDest  isFraud  isFlaggedFraud  error_orig  error_dest  \
0             0.0        0               0         0.0     9839.64   
1             0.0        0               0         0.0     1864.28   
2             0.0        1               0         0.0      181.00   
3             0.0        1               0         0.0    21363.00   
4             0.0        0               0         0.0    11668.14   

   type_CASH_IN  type_CASH_OUT  type_DEBIT  type_PAYMENT  type_TRANSFER  
0         False          False       False          True          False  
1         False       

In [8]:
# 1. Separate the data into 'Features' (X) and 'Target' (y)
X = df.drop(['isFraud', 'isFlaggedFraud'], axis=1) # Everything except the fraud status
y = df['isFraud']              # Just the fraud status
# Drop both the target AND the column that isn't a feature
# 2. Split the data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize and train the model
model = XGBClassifier()
model.fit(X_train, y_train)

print("Model training complete!")

Model training complete!


In [9]:
# Use the model to predict fraud on the test data (the "final exam")
y_pred = model.predict(X_test)

# See how the model performed
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19978
           1       1.00      0.95      0.98        22

    accuracy                           1.00     20000
   macro avg       1.00      0.98      0.99     20000
weighted avg       1.00      1.00      1.00     20000



In [10]:
import joblib

# Save the model to a file
joblib.dump(model, '../data/fraud_model.pkl')
print("Model saved successfully!")


Model saved successfully!
